In [0]:
# List all tables in samples.bakehouse schema
print("Tables in samples.bakehouse:")
tables = spark.catalog.listTables("samples.bakehouse")
for table in tables:
    print(f"  - {table.name}")

print("\n" + "="*50 + "\n")


In [0]:
#Read the data from sales_customers, sales_franchises and sales_transactions tables
sales_customers = spark.read.table("samples.bakehouse.sales_customers")
sales_franchises = spark.read.table("samples.bakehouse.sales_franchises")
sales_transactions = spark.read.table("samples.bakehouse.sales_transactions")
sales_suppliers = spark.read.table("samples.bakehouse.sales_suppliers")

In [0]:
#Show Schema of the tables
print("sales_customers schema")
sales_customers.printSchema()


In [0]:
print("sales_franchises schema")
sales_franchises.printSchema()


In [0]:
print("sales_transactions schema")
sales_transactions.printSchema()


In [0]:
print("sales_suppliers schema")
sales_suppliers.printSchema()

In [0]:
# Join transactions with the store information
enriched_transactions = sales_franchises.join(sales_transactions,on="franchiseID",how="inner")
display(enriched_transactions)

# This is a basic join when the columns are named equal across the tables

In [0]:
# The "on" clause can contain more complex expressions
# Here we join on the store name, which is different across the tables
enriched_transactions = sales_franchises.join(sales_transactions, on=sales_transactions.franchiseID == sales_franchises.franchiseID, how="inner")
display(enriched_transactions)

#This is useful when the columns are named differently across the tables, but the values are the same. As it works in SQL


In [0]:
# When doing joins, it brings all of the column from each entity, it is advided for analytics to bring only what you need
# A best practice is to rename a column to disambiguate 
import pyspark.sql.functions as F
from pyspark.sql.functions import col
enriched_transactions = sales_franchises.select(
    "franchiseID",
    col("name").alias("store_name"),
    col("city").alias("city_name"),
    col("country").alias("country_name")
    ).\
join(sales_transactions, on=sales_transactions.franchiseID == sales_franchises.franchiseID, how="inner")
display(enriched_transactions)


In [0]:
#The demo says to analyze the relationship between franchises and suppliers using full outer join
import pyspark.sql.functions as F 
from pyspark.sql.functions import col
full_outer_join = sales_franchises.withColumnRenamed("name", "franchise_name")\
    .join(sales_suppliers.select(\
        "supplierID",
        col("name").alias("supplier_name")),on="supplierID",how="full_outer")
display(enriched_transactions)

non_matching_results = full_outer_join.filter(
    col("franchise_name").isNull() |
    col("supplier_name").isNull()   
)\
.select("franchiseID", "franchise_name",col("supplierID").alias("orphaned_supplier_id"))
display(non_matching_results)

In [0]:
#Create temp views for SQL 

sales_franchises.createOrReplaceTempView("franchises")
sales_suppliers.createOrReplaceTempView("suppliers")

In [0]:
%sql
--Do the same analysis but in SQL

SELECT f.franchiseID, f.name AS franchise_name
FROM franchises f 
FULL OUTER JOIN suppliers s ON f.supplierID = s.supplierID
WHERE f.supplierID IS NULL OR s.supplierID IS NULL

In [0]:
#Identify supplierID in each dataframe
franchise_suppliers = sales_franchises.select("supplierID").distinct()
all_suppliers = sales_suppliers.select("supplierID").distinct()

#Find suppliers in franchises but not in suppliers (invalid suppliers)
invalid_suppliers = franchise_suppliers.subtract(all_suppliers).alias("invalid_suppliers")
display(invalid_suppliers)